In [ ]:
!pip install tsfresh tsfel xgboost lightgbm imblearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.4/235.4 kB 13.6 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import re

from tsfresh import extract_features, select_features
from tsfresh.utilities.dataframe_functions import impute
import tsfel

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from imblearn.over_sampling import SMOTE

# TSFRESH

## Load dataset

In [ ]:
train = "https://raw.githubusercontent.com/HKRM1006/data/main/A101_train_preprocessed.csv"
test = "https://raw.githubusercontent.com/HKRM1006/data/main/A101_test_preprocessed.csv"
train_dataset = pd.read_csv(train)
test_dataset = pd.read_csv(test)
train_dataset.head()

,Datetime,Pressure,Solar_Radiation,Temperature,Dew_Point,Humidity,Wind_Direction,Wind_Gust,Wind_Speed,Rain
0,2017-02-18 15:00:00,1.162247,1.868269,0.677010,-0.075491,-0.668846,-1.018556,1.576822,0.686933,0
1,2017-02-18 16:00:00,0.889424,1.548854,1.361493,-1.041062,-1.450878,-0.964861,0.587912,0.259384,0
2,2017-02-18 17:00:00,0.389248,2.058422,1.469569,-0.669689,-1.379784,-0.460122,0.752730,2.824683,0
3,2017-02-18 18:00:00,-0.292809,2.139611,0.857137,-0.224041,-0.882128,-1.050773,1.961398,0.259384,0
4,2017-02-18 19:00:00,-0.929396,1.341608,1.865849,-1.041062,-1.735253,-0.954121,0.203335,-0.453200,0


## Gộp các dòng thành các time window riêng biệt

In [ ]:
def make_time_window(df, window_size):
    X_list = []
    y_list = []
    feature_cols = [c for c in df.columns if c not in ["Datetime", "Rain"]]
    for i in range(len(df) - window_size):
        # Lấy cửa sổ dữ liệu và gán id
        window = df.iloc[i : i + window_size][feature_cols].copy()
        window['id'] = i
        # Lưu nhãn của dòng tiếp theo sau cửa sổ
        label = df.iloc[i + window_size]["Rain"]
        X_list.append(window)
        y_list.append(label)
    X = pd.concat(X_list)
    y = pd.Series(y_list, index=range(len(y_list)))
    return X, y
X_train , y_train = make_time_window(train_dataset, 12)
X_test , y_test = make_time_window(test_dataset, 12)


## Sử dụng TSFRESH cho feature extraction

In [ ]:
# Feature extract
X_train_feat = extract_features(X_train, column_id="id")
X_test_feat = extract_features(X_test, column_id="id)
# Chọn các feature có ý nghĩa thống kê
X_train_feat = impute(X_train_feat)
X_test_feat = impute(X_test_feat)
X_train_tsfresh = select_features(X_train_feat, y_train)
X_test_tsfresh = X_test_feat[X_train_tsfresh.columns]


Feature Extraction: 100%|██████████| 110/110 [01:58<00:00,  1.08s/it]
/usr/local/lib/python3.12/dist-packages/tsfresh/utilities/dataframe_functions.py:198: RuntimeWarning: The columns ['Pressure__partial_autocorrelation__lag_6'
 'Pressure__partial_autocorrelation__lag_7'
 'Pressure__partial_autocorrelation__lag_8' ...
 'Wind_Speed__agg_linear_trend__attr_"stderr"__chunk_len_50__f_agg_"mean"'
 'Wind_Speed__agg_linear_trend__attr_"stderr"__chunk_len_50__f_agg_"var"'
 'Wind_Speed__query_similarity_count__query_None__threshold_0.0'] did not have any finite values. Filling with zeros.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/tsfresh/utilities/dataframe_functions.py:198: RuntimeWarning: The columns ['Wind_Speed__partial_autocorrelation__lag_6'
 'Wind_Speed__partial_autocorrelation__lag_7'
 'Wind_Speed__partial_autocorrelation__lag_8' ...
 'Wind_Gust__agg_linear_trend__attr_"stderr"__chunk_len_50__f_agg_"mean"'
 'Wind_Gust__agg_linear_trend__attr_"stderr"__chunk_len_50__f_agg_

# TsFEL

In [ ]:
train = "https://raw.githubusercontent.com/HKRM1006/data/main/A101_train_preprocessed.csv"
test = "https://raw.githubusercontent.com/HKRM1006/data/main/A101_test_preprocessed.csv"
train_dataset = pd.read_csv(train)
test_dataset = pd.read_csv(test)
def tsfel_feature_extract(df, window_size=12):
    X_windows = []
    y_list = []
    feature_cols = [c for c in df.columns if c not in ["Datetime", "Rain"]]
    for i in range(len(df) - window_size):
        window = df.iloc[i : i + window_size][feature_cols].copy()
        X_windows.append(window)
        y_list.append(df.iloc[i + window_size]["Rain"])
    cfg = tsfel.get_features_by_domain()
    cfg = {
        "statistical": cfg["statistical"],
        "temporal": cfg["temporal"]
    }
    X_features = tsfel.time_series_features_extractor(cfg, X_windows, fs=1)
    return X_features, np.array(y_list)
X_train_tsfel, y_train_tsfel = tsfel_feature_extract(train_dataset)
X_test_tsfel, y_test_tsfel = tsfel_feature_extract(test_dataset)

# Thử nghiệm trên model

In [ ]:
# Sửa feature name cho LGBM
def clean_feature_names(df):
    df = df.copy()
    df.columns = [
        re.sub(r'[^A-Za-z0-9_]+', '_', col)
        for col in df.columns
    ]
    return df
X_train_tsfresh = clean_feature_names(X_train_tsfresh)
X_test_tsfresh = clean_feature_names(X_test_tsfresh)
imputer = SimpleImputer(strategy="median")
X_train_tsfel = clean_feature_names(X_train_tsfel)
X_test_tsfel = clean_feature_names(X_test_tsfel)
X_train_tsfel = pd.DataFrame(
    imputer.fit_transform(X_train_tsfel),
    columns=X_train_tsfel.columns
)
X_test_tsfel = pd.DataFrame(
    imputer.transform(X_test_tsfel),
    columns=X_test_tsfel.columns
)
models = {
    "RF": RandomForestClassifier(),
    "SVM": SVC(),
    "NB": GaussianNB(),
    "KNN": KNeighborsClassifier(),
    "LR": LogisticRegression(max_iter=1000),
    "XGB": XGBClassifier(eval_metric='logloss'),
    "LGBM": LGBMClassifier()
}
def evaluate(models, X_train, X_test, y_train, y_test):
    results = {}
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    for name, model in models.items():
        model.fit(X_train_res, y_train_res)
        y_pred = model.predict(X_test)
        results[name] = {
            "Accuracy": accuracy_score(y_test, y_pred),
            "F1": f1_score(y_test, y_pred, average="weighted"),
            "Precision": precision_score(y_test, y_pred, average="weighted"),
            "Recall": recall_score(y_test, y_pred, average="weighted")
        }
    return pd.DataFrame(results).T
res_tsfresh = evaluate(models, X_train_tsfresh, X_test_tsfresh, y_train, y_test)
res_tsfel = evaluate(models, X_train_tsfel, X_test_tsfel, y_train, y_test)
print("TSFRESH result")
print(res_tsfresh)
print("TSFEL result")
print(res_tsfel)
res_tsfresh["Feature"] = "TSFRESH"
res_tsfel["Feature"] = "TSFEL"
result = pd.concat([res_tsfresh, res_tsfel])
result.to_csv("baseline_result.csv")
from google.colab import files
files.download("baseline_result.csv")
from google.colab import runtime
runtime.unassign()

[LightGBM] [Info] Number of positive: 27160, number of negative: 27160
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.127254 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 435223
[LightGBM] [Info] Number of data points in the train set: 54320, number of used features: 1707
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[LightGBM] [Info] Number of positive: 27160, number of negative: 27160
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013661 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 67144
[LightGBM] [Info] Number of data points in the train set: 54320, number of used features: 272
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
TSFRESH result
      Accuracy        F1  Precision    Recall
RF    0.927151  0.911943   0.912514  0.927151
SVM   0.100082  0.050786   0.854036  0.100082
NB    0.099537  0.049022   0.868086  0.099537
KNN   0.714325  0.776352   0.884582  0.714325
LR    0.759668  0.801092   0.854211  0.759668
XGB   0.927424  0.912867   0.913049  0.927424
LGBM  0.930828  0.918521   0.918833  0.930828
TSFEL result
      Accuracy        F1  Precision    Recall
RF    0.923203  0.909189   0.906509  0.923203
SVM   0.724265  0.784724   0.899704  0.724265
NB    0.479575  0.580404 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>